In [ ]:
# Setup
from pathlib import Path
import sys, json
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np

from data import load_data, TRUE_ELASTICITY, UNIT_COSTS
from model import estimate_elasticity, find_optimal_price, find_optimal_profit_price

ARTEFACTS_DIR = PROJECT_ROOT / 'artefacts'
with open(ARTEFACTS_DIR / 'pricing_results.json') as f:
    results = json.load(f)
print("Overall estimated elasticity:", results['overall_elasticity']['elasticity'])
print("Ground-truth elasticity:", results['true_elasticity'])

In [ ]:
# 1. Validate the estimator against the ground truth
est_elast = results['overall_elasticity']['elasticity']
true_elast = results['true_elasticity']
abs_err = abs(est_elast - true_elast)
pct_err = abs_err / abs(true_elast) * 100
print(f"Estimated: {est_elast:.4f}")
print(f"True:      {true_elast}")
print(f"Abs error: {abs_err:.4f}")
print(f"Pct error: {pct_err:.2f}%")
print()
print("Per-tier estimates:")
print(pd.DataFrame(results['tier_elasticities']).to_string(index=False))

In [ ]:
# 2. Compare revenue-max vs profit-max pricing
optima_df = pd.DataFrame(results['optima_by_tier'])
optima_df['rev_minus_profit_price'] = optima_df['rev_optimal_price'] - optima_df['profit_optimal_price']
print(optima_df[['tier', 'unit_cost', 'elasticity', 'rev_optimal_price',
                  'profit_optimal_price', 'rev_minus_profit_price']].round(2).to_string(index=False))

In [ ]:
# 3. Business-readable summary
lines = [
    "=== BUSINESS SUMMARY ===",
    "Business question: given historical transaction data and a unit cost, what is the",
    "optimal price?",
    "",
    f"Estimated elasticity (overall): {est_elast:.3f}",
    f"Ground-truth elasticity:        {true_elast}",
    f"Estimation error:                {pct_err:.2f}%",
    "",
    "Per-tier optimal prices:",
]
for row in results['optima_by_tier']:
    lines.append(
        f"  {row['tier']:>10s}  cost=${row['unit_cost']:.2f}  "
        f"rev_max=${row['rev_optimal_price']:.2f}  "
        f"profit_max=${row['profit_optimal_price']:.2f}"
    )

lines += [
    "",
    "Recommended action:",
    "- Use the profit-maximising price as the default recommendation (revenue-max ignores cost).",
    "- For premium tiers, the revenue-max price is materially higher — useful for top-line growth campaigns.",
    "- Re-estimate elasticity weekly as new transaction data arrives; elasticity can drift seasonally.",
]
print("\n".join(lines))